# CRIVO — Pointer/Copy Causal v2 (auditado)

**Pergunta experimental:** adicionar uma via explícita de cópia melhora a preservação de spans arbitrários **sem alterar nenhum peso do Transformer** e sem degradar a semântica já existente?

Este notebook foi refeito para evitar os bugs das versões anteriores:

- usa o `LinguagemProfunda` real do repositório;
- localiza automaticamente o checkpoint `05_05_dialogo_estado`;
- copia a base para SSD local;
- congela 100% do Transformer/embedding/lm-head amarrado;
- a pointer head só pode copiar posições do **prompt original**, nunca tokens já gerados;
- o prompt segue exatamente `<usuario> ... <fim> <assistente>`;
- train/dev/blind vêm de um único pool global e são disjuntos;
- o gerador de nomes é finito — **não há loop infinito**;
- blind = 100 entidades × 6 tarefas = 600 casos, congelado antes do treino;
- splits balanceados: metade `close` / metade `distant`, comprimentos 1–4 tokens distribuídos por cota (a versão anterior deixava o blind ~98% `distant` e ~97% com 4 tokens);
- há exemplos `no-copy` para proteger `p_gen`;
- sanity/smoke test executa backward + 5 steps + geração antes do treino real;
- fingerprint SHA-256 dos pesos da base é verificado antes, durante e depois;
- mesmo que todos os gates passem: `ativo_no_chat=false` e `promovido=false`.

> Use **Ambiente de execução → Executar tudo** em um Colab com GPU.


In [ ]:
# Célula 0 — ambiente, GPU, Drive, dependências
import os, sys, gc, json, math, time, hashlib, shutil, traceback, re, itertools, copy, random
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("PRECHECK 0/8 — ambiente", flush=True)

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except Exception as exc:
    raise SystemExit(f"ERRO: PyTorch indisponível: {exc}")

try:
    from google.colab import drive
except Exception as exc:
    raise SystemExit(
        "ERRO: este notebook foi preparado para Google Colab. "
        f"google.colab não está disponível: {exc}"
    )

if not torch.cuda.is_available():
    raise SystemExit(
        "ERRO: GPU não disponível. No Colab: Ambiente de execução → Alterar tipo de ambiente → GPU."
    )

print(f"GPU OK: {torch.cuda.get_device_name(0)}", flush=True)
print(f"PyTorch={torch.__version__} CUDA={torch.version.cuda}", flush=True)
torch.backends.cuda.matmul.allow_tf32 = True

print("Montando Google Drive...", flush=True)
drive.mount("/content/drive", force_remount=False)
print("Drive montado.", flush=True)

try:
    import tokenizers
    from tokenizers import Tokenizer
except Exception:
    import subprocess
    print("Instalando tokenizers...", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tokenizers>=0.20,<1"])
    import tokenizers
    from tokenizers import Tokenizer

print(f"tokenizers={tokenizers.__version__}", flush=True)

SEED = 20261012
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("PRECHECK 0/8 OK", flush=True)


In [ ]:
# Célula 1 — clone limpo da main e import da arquitetura real
import subprocess

print("PRECHECK 1/8 — repositório", flush=True)
REPO_URL = "https://github.com/HROSONE/CRIVO"
REPO_DIR = Path("/content/CRIVO")

# Sempre parte de clone limpo para não depender de células antigas do runtime.
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

r = subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
    capture_output=True, text=True
)
if r.returncode != 0:
    raise SystemExit(f"ERRO ao clonar {REPO_URL}:\nSTDOUT:\n{r.stdout}\nSTDERR:\n{r.stderr}")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

try:
    import linguagem_profunda as lp
except Exception as exc:
    raise SystemExit(f"ERRO ao importar linguagem_profunda.py: {exc}\n{traceback.format_exc()}")

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()

esperados = ["<pad>", "<documento>", "<usuario>", "<assistente>", "<fim>"]
if list(lp.ESPECIAIS) != esperados:
    raise SystemExit(
        f"ERRO: especiais do repo mudaram. Esperado={esperados}, encontrado={list(lp.ESPECIAIS)}"
    )

print("commit main:", commit, flush=True)
print("ESPECIAIS:", lp.ESPECIAIS, flush=True)
print("PRECHECK 1/8 OK", flush=True)


In [ ]:
# Célula 2 — localizar fase 5 mais recente e copiar base para SSD
print("PRECHECK 2/8 — checkpoint", flush=True)

DRIVE_CHECKPOINTS = Path("/content/drive/MyDrive/CRIVO/checkpoints")
ESCOLA = DRIVE_CHECKPOINTS / "escola_curricular_longa"

if not ESCOLA.is_dir():
    raise SystemExit(f"ERRO: não encontrei {ESCOLA}")

candidatos = []
for run in sorted(ESCOLA.glob("run_*")):
    fase = run / "05_05_dialogo_estado"
    pesos = fase / "pesos.pt"
    tok_file = fase / "tokenizer.json"
    if pesos.is_file() and tok_file.is_file():
        try:
            stamp = pesos.stat().st_mtime
        except OSError:
            stamp = 0
        candidatos.append((stamp, run, fase))

if not candidatos:
    raise SystemExit(
        "ERRO: nenhum run_* contém 05_05_dialogo_estado/pesos.pt + tokenizer.json"
    )

candidatos.sort(key=lambda x: (x[0], x[1].name))
_, BASE_RUN, ORIGEM = candidatos[-1]

LOCAL = Path("/content/crivo_pointer_v2_base")
if LOCAL.exists():
    shutil.rmtree(LOCAL)
LOCAL.mkdir(parents=True)

print("Base escolhida:", BASE_RUN, flush=True)
for nome in ("pesos.pt", "tokenizer.json"):
    src = ORIGEM / nome
    dst = LOCAL / nome
    print(f"Copiando {nome} para SSD...", flush=True)
    shutil.copy2(src, dst)
    if not dst.is_file() or dst.stat().st_size != src.stat().st_size:
        raise SystemExit(f"ERRO: cópia incompleta de {nome}")

print(f"pesos.pt local: {(LOCAL/'pesos.pt').stat().st_size/1024**2:.2f} MiB", flush=True)
print("PRECHECK 2/8 OK", flush=True)


In [ ]:
# Célula 3 — carregar, congelar e fingerprint da base
print("PRECHECK 3/8 — modelo/tokenizer", flush=True)

try:
    modelo, tokenizer, estado = lp.carregar(LOCAL, dispositivo="cpu")
except Exception as exc:
    raise SystemExit(f"ERRO ao carregar checkpoint: {exc}\n{traceback.format_exc()}")

cfg = modelo.config
if cfg.vocabulario != 8192 or cfg.dimensao != 384 or cfg.camadas != 8 or cfg.cabecas != 8:
    print(
        "AVISO: configuração difere do snapshot esperado "
        f"(vocab={cfg.vocabulario}, dim={cfg.dimensao}, camadas={cfg.camadas}, cabecas={cfg.cabecas}).",
        flush=True,
    )

def sha256_arquivo(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

def fingerprint_modulo(mod: nn.Module) -> str:
    h = hashlib.sha256()
    with torch.no_grad():
        for nome, param in sorted(mod.named_parameters(), key=lambda x: x[0]):
            t = param.detach().cpu().contiguous()
            h.update(nome.encode("utf-8"))
            h.update(str(t.dtype).encode("ascii"))
            h.update(str(tuple(t.shape)).encode("ascii"))
            h.update(t.numpy().tobytes())
    return h.hexdigest()

modelo.eval()
for p in modelo.parameters():
    p.requires_grad_(False)
    p.grad = None

N_BASE = sum(p.numel() for p in modelo.parameters())
HASH_ANTES = fingerprint_modulo(modelo)
SHA_PESOS = sha256_arquivo(LOCAL / "pesos.pt")
SHA_TOKENIZER = sha256_arquivo(LOCAL / "tokenizer.json")

ids_especiais = {}
for s in lp.ESPECIAIS:
    tid = tokenizer.token_to_id(s)
    if tid is None:
        raise SystemExit(f"ERRO: especial ausente no tokenizer: {s}")
    ids_especiais[s] = int(tid)

ID_PAD = ids_especiais["<pad>"]
ID_USUARIO = ids_especiais["<usuario>"]
ID_ASSISTENTE = ids_especiais["<assistente>"]
ID_FIM = ids_especiais["<fim>"]
ESPECIAIS_SET = set(ids_especiais.values())
PROIBIDOS_GERACAO = [i for i in ESPECIAIS_SET if i != ID_FIM]

dispositivo = torch.device("cuda")
modelo = modelo.to(dispositivo).eval()

print(
    f"Config: vocab={cfg.vocabulario} dim={cfg.dimensao} camadas={cfg.camadas} "
    f"cabecas={cfg.cabecas} contexto={cfg.contexto} dropout={cfg.dropout}",
    flush=True
)
print(f"Base: {N_BASE:,} parâmetros — todos congelados", flush=True)
print("IDs especiais:", ids_especiais, flush=True)
print("base_hash_before:", HASH_ANTES, flush=True)
print("pesos.pt sha256:", SHA_PESOS, flush=True)
print("tokenizer sha256:", SHA_TOKENIZER, flush=True)
print("PRECHECK 3/8 OK", flush=True)


In [ ]:
# Célula 4 — pointer/copy separado; fonte de cópia = SOMENTE prompt original
print("PRECHECK 4/8 — pointer", flush=True)

@torch.no_grad()
def estados_congelados(ids: torch.Tensor) -> torch.Tensor:
    if ids.ndim != 2:
        raise ValueError("ids deve ser [B,T]")
    if ids.shape[1] > cfg.contexto:
        raise ValueError(f"sequência {ids.shape[1]} > contexto {cfg.contexto}")
    pos = torch.arange(ids.shape[1], device=ids.device)
    x = modelo.embedding(ids) + modelo.posicao(pos)
    for bloco in modelo.blocos:
        x = bloco(x)
    return modelo.norm(x)

def mascara_fonte(ids: torch.Tensor, prompt_lens: torch.Tensor) -> torch.Tensor:
    """
    True somente nas posições do prompt ORIGINAL e que não são especiais.
    Tokens da resposta (teacher-forced ou gerados) nunca entram na fonte de cópia.
    """
    b, t = ids.shape
    pos = torch.arange(t, device=ids.device).unsqueeze(0)
    src = pos < prompt_lens.unsqueeze(1)
    for sid in ESPECIAIS_SET:
        src &= ids.ne(sid)
    return src

class PointerCopy(nn.Module):
    def __init__(self, dimensao: int, vocabulario: int, rank: int = 32, p_gen_init: float = 0.90):
        super().__init__()
        if not (0.0 < p_gen_init < 1.0):
            raise ValueError("p_gen_init deve estar entre 0 e 1")
        self.rank = int(rank)
        self.vocabulario = int(vocabulario)
        self.q = nn.Linear(dimensao, rank, bias=False)
        self.k = nn.Linear(dimensao, rank, bias=False)
        self.gate = nn.Linear(dimensao, 1)
        nn.init.normal_(self.q.weight, std=0.02)
        nn.init.normal_(self.k.weight, std=0.02)
        nn.init.zeros_(self.gate.weight)
        nn.init.constant_(self.gate.bias, math.log(p_gen_init / (1.0 - p_gen_init)))

    def forward(self, h: torch.Tensor, ids: torch.Tensor, source_mask: torch.Tensor):
        b, t, _ = h.shape
        if ids.shape != (b, t) or source_mask.shape != (b, t):
            raise ValueError("shape incompatível em PointerCopy.forward")

        q = self.q(h)
        k = self.k(h)
        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.rank)

        causal = torch.tril(torch.ones((t, t), dtype=torch.bool, device=h.device))
        valido = causal.unsqueeze(0) & source_mask.unsqueeze(1)  # [B,T,T]

        # -1e4 evita NaN em linhas sem nenhuma fonte; depois essas linhas são zeradas.
        scores = scores.masked_fill(~valido, -1e4)
        attn = torch.softmax(scores, dim=-1)
        tem_fonte = valido.any(dim=-1, keepdim=True)
        attn = torch.where(tem_fonte, attn, torch.zeros_like(attn))
        attn = attn * valido.to(attn.dtype)
        attn = attn / attn.sum(dim=-1, keepdim=True).clamp_min(1e-8)

        p_copy = torch.zeros(
            (b, t, self.vocabulario), dtype=h.dtype, device=h.device
        )
        ids_expand = ids.unsqueeze(1).expand(b, t, t)
        p_copy.scatter_add_(2, ids_expand, attn)

        # Defesa redundante: nenhum especial pode receber massa de copy.
        p_copy[:, :, list(ESPECIAIS_SET)] = 0.0
        soma = p_copy.sum(dim=-1, keepdim=True)
        p_copy = torch.where(
            soma > 0,
            p_copy / soma.clamp_min(1e-8),
            torch.zeros_like(p_copy),
        )

        p_gen = torch.sigmoid(self.gate(h))
        return p_copy, p_gen, attn, tem_fonte

def distribuicao_final(pointer_obj: PointerCopy, ids: torch.Tensor, prompt_lens: torch.Tensor):
    modelo.eval()
    with torch.no_grad():
        h = estados_congelados(ids)
        logits = F.linear(h, modelo.embedding.weight)
        p_vocab = torch.softmax(logits, dim=-1)
        if PROIBIDOS_GERACAO:
            p_vocab = p_vocab.clone()
            p_vocab[:, :, PROIBIDOS_GERACAO] = 0.0
            p_vocab = p_vocab / p_vocab.sum(dim=-1, keepdim=True).clamp_min(1e-8)

    h = h.detach()
    src = mascara_fonte(ids, prompt_lens)
    p_copy, p_gen, attn, tem_fonte = pointer_obj(h, ids, src)

    # Sem nenhuma fonte copiável, força a rota vocab (não renormaliza um vetor zero).
    p_gen_eff = torch.where(tem_fonte, p_gen, torch.ones_like(p_gen))
    p_final = p_gen_eff * p_vocab + (1.0 - p_gen_eff) * p_copy
    p_final = p_final / p_final.sum(dim=-1, keepdim=True).clamp_min(1e-8)
    return p_final, p_vocab, p_copy, p_gen, p_gen_eff, attn, src

RANK = 32
pointer = PointerCopy(cfg.dimensao, cfg.vocabulario, rank=RANK).to(dispositivo)
N_POINTER = sum(p.numel() for p in pointer.parameters() if p.requires_grad)

# Teste estrutural independente de dados reais.
_b, _t = 2, 7
_h = torch.randn(_b, _t, cfg.dimensao, device=dispositivo)
_ids = torch.randint(5, cfg.vocabulario, (_b, _t), device=dispositivo)
_ids[:, 0] = ID_USUARIO
_pm = torch.tensor([5, 4], device=dispositivo)
_src = mascara_fonte(_ids, _pm)
_pc, _pg, _a, _tf = pointer(_h, _ids, _src)
assert _pc.shape == (_b, _t, cfg.vocabulario)
assert _a.shape == (_b, _t, _t)
assert torch.isfinite(_pc).all() and torch.isfinite(_a).all()
assert float(_pc[:, :, list(ESPECIAIS_SET)].abs().max().item()) == 0.0
del _h, _ids, _pm, _src, _pc, _pg, _a, _tf

print(f"Pointer/gate treinável: {N_POINTER:,} parâmetros ({100*N_POINTER/N_BASE:.4f}% da base)", flush=True)
print(f"p_gen inicial={torch.sigmoid(pointer.gate.bias.detach()).item():.4f}", flush=True)
print("PRECHECK 4/8 OK", flush=True)


In [ ]:
# Célula 5 — pool GLOBAL finito + split sem reposição (sem prefixos de split)
print("PRECHECK 5/8 — entidades/splits", flush=True)

rng = random.Random(SEED)

CATEGORIAS = ["recipiente", "ferramenta", "instrumento", "veiculo", "alimento"]
CAT_TXT = {
    "recipiente": "recipiente",
    "ferramenta": "ferramenta",
    "instrumento": "instrumento",
    "veiculo": "veículo",
    "alimento": "alimento",
}
FUNCOES_TXT = {
    "recipiente": "guardar substâncias",
    "ferramenta": "apertar peças",
    "instrumento": "medir sinais",
    "veiculo": "transportar carga",
    "alimento": "nutrir pessoas",
}
MATERIAIS = ["madeira", "metal", "vidro", "tecido", "argila", "bambu"]

SILABAS = [
    "ba","be","bi","bo","bu","ca","ce","ci","co","cu","da","de","di","do","du",
    "fa","fe","fi","fo","ga","ge","gi","go","la","le","li","lo","lu","ma","me",
    "mi","mo","mu","na","ne","ni","no","nu","pa","pe","pi","po","pu","ra","re",
    "ri","ro","ru","sa","se","si","so","su","ta","te","ti","to","tu","va","ve",
    "vi","vo","za","ze","zi","zo"
]
RAROS = [
    "xz","zq","qv","jk","wq","vx","jq","zx","qk","wv","xq","jz","vw","kx","xv",
    "jw","qx","kz","wj","zv","bq","cq","dq","fq","gq","hq","mq","pq","tq","xb",
    "xd","xf","xg","xh","xj","xk","xl","xm","xn","xp","xr","xs","xt","xw","zb",
    "zd","zf","zg","zh","zj","zk","zl","zm","zn","zp","zr","zs","zt","zw","vb",
    "vd","vf","vg","vh","vj","vk","vl","vm","vn","vp","vr","vs","vt","vw"
]

def enc(texto: str):
    return list(tokenizer.encode(texto, add_special_tokens=False).ids)

def candidato_valido(nome: str):
    if not re.fullmatch(r"[A-Za-z][A-Za-z0-9-]{2,19}", nome):
        return None
    ids = enc(nome)
    if not (1 <= len(ids) <= 4):
        return None
    if any(i in ESPECIAIS_SET for i in ids):
        return None
    # round-trip estrito: o nome realmente é representável como esse span.
    if tokenizer.decode(ids).strip() != nome:
        return None
    return ids

def universo_candidatos():
    """
    Universo FINITO. Não há while.
    Retorna pares (nome, grupo) na mesma distribuição para todos os splits.
    """
    out = []

    # 1) Tokens unitários seguros do próprio vocab garantem cobertura real de 1 token.
    for tid in range(cfg.vocabulario):
        if tid in ESPECIAIS_SET:
            continue
        s = tokenizer.decode([tid]).strip()
        if re.fullmatch(r"[A-Za-z][A-Za-z0-9-]{2,15}", s or ""):
            # remove palavras óbvias das tarefas para não transformar categorias em "nomes"
            if s.casefold() in {x.casefold() for x in CATEGORIAS + list(CAT_TXT.values())}:
                continue
            raridade = sum(ch.lower() in "qwxzjkv" for ch in s)
            grupo = "distant" if raridade >= 2 else "close"
            out.append((s, grupo))

    # 2) Close: combinações silábicas, com algumas variações de formato.
    for i, a in enumerate(SILABAS):
        for j, b in enumerate(SILABAS):
            base = a + b
            out.append((base, "close"))
            if (i + j) % 7 == 0:
                out.append((base.capitalize() + str((i*17+j) % 100), "close"))
            if (i + j) % 11 == 0:
                out.append((a + "-" + b + str((i+j) % 10), "close"))

    # 3) Distant: espaço combinatório grande, porém ainda finito.
    for i, a in enumerate(RAROS):
        for j, b in enumerate(RAROS):
            out.append((a + b, "distant"))
            if (i + j) % 3 == 0:
                out.append((a + str((i*37+j) % 1000), "distant"))
            if (i + j) % 5 == 0:
                out.append((a + "-" + b, "distant"))

    # 4) Mais formatos curtos para diversidade sem depender de tentativas aleatórias.
    letras_raras = "qwxzjkv"
    for i, a in enumerate(letras_raras):
        for j, b in enumerate("aeiou"):
            for d in range(10):
                out.append((a + b + str(d) + letras_raras[(i+j+d) % len(letras_raras)], "distant"))

    return out

vistos = set()
pool = []
rejeitados = Counter()

for nome, grupo in universo_candidatos():
    chave = nome.casefold()
    if chave in vistos:
        rejeitados["duplicado"] += 1
        continue
    ids = candidato_valido(nome)
    if ids is None:
        rejeitados["tokenizacao"] += 1
        continue
    vistos.add(chave)
    pool.append({
        "nome": nome,
        "ids": ids,
        "n_tokens": len(ids),
        "grupo": grupo,
    })

if len(pool) < 660:
    raise SystemExit(f"ERRO: pool válido tem {len(pool)} nomes; precisamos de 660.")

cont_pool = Counter((x["grupo"], x["n_tokens"]) for x in pool)
print("Pool válido:", len(pool), "rejeições:", dict(rejeitados), flush=True)
print("Pool grupo×tokens:", dict(sorted(cont_pool.items())), flush=True)

if not any(x["grupo"] == "distant" for x in pool):
    raise SystemExit("ERRO: pool não contém grupo distant.")
if not any(x["grupo"] == "close" for x in pool):
    raise SystemExit("ERRO: pool não contém grupo close.")

cont_len = Counter(x["n_tokens"] for x in pool)
for L in (1, 2, 3, 4):
    if cont_len.get(L, 0) < 3:
        raise SystemExit(
            f"ERRO: precisamos de pelo menos 3 nomes de {L} token(s) "
            f"(um por split); encontrados={cont_len.get(L, 0)}."
        )
for grupo in ("close", "distant"):
    if sum(x["grupo"] == grupo for x in pool) < 3:
        raise SystemExit(f"ERRO: grupo {grupo} insuficiente para os três splits.")

# Partição global estratificada por (grupo, n_tokens), sem reposição.
# Com o tokenizer real o pool é muito desigual (ex.: ~7 mil "distant" de 4 tokens
# e só 2 "distant" de 1 token). Tirar sempre do maior bucket deixava o blind ~98%
# distant / ~97% de 4 tokens, o que esvaziava os gates close×distant e por comprimento.
# Aqui: metade close / metade distant; dentro de cada grupo, cota igual por
# comprimento (bucket pequeno entrega tudo e a sobra vai para os outros
# comprimentos do mesmo grupo); cada célula é repartida entre train/dev/blind
# na proporção 480/80/100 com maior resto. Tudo determinístico, sem while.
def particionar_estratificado(pool, n_train=480, n_dev=80, n_blind=100):
    total = n_train + n_dev + n_blind
    if total > len(pool):
        raise SystemExit(f"ERRO: pool={len(pool)} < total necessário={total}")

    comprimentos = sorted({x["n_tokens"] for x in pool})
    if comprimentos != [1, 2, 3, 4]:
        raise SystemExit(
            "ERRO: o pool não cobre explicitamente 1,2,3,4 tokens. "
            f"Cobertura encontrada={comprimentos}. O experimento para aqui."
        )

    buckets = defaultdict(list)
    for item in pool:
        buckets[(item["grupo"], item["n_tokens"])].append(dict(item))

    rr = random.Random(SEED + 1001)
    for chave in sorted(buckets):
        rr.shuffle(buckets[chave])

    def encher(disponivel, alvo):
        """Cota igual por chave; chave com pouca oferta entrega tudo e repassa a sobra."""
        chaves = sorted(disponivel, key=lambda k: (disponivel[k], k))
        cota, restante = {}, alvo
        for idx, k in enumerate(chaves):
            faltam_chaves = len(chaves) - idx
            parte = restante if faltam_chaves == 1 else restante // faltam_chaves
            cota[k] = min(disponivel[k], parte)
            restante -= cota[k]
        if restante:
            raise SystemExit(f"ERRO: oferta insuficiente para cota {alvo}: {disponivel}")
        return cota

    def repartir(cotas, n):
        """Divide n entre as células proporcionalmente às cotas (maior resto)."""
        soma = sum(cotas.values())
        ideal = {k: cotas[k] * n / soma for k in cotas}
        out = {k: int(ideal[k]) for k in cotas}
        resto = n - sum(out.values())
        for k in sorted(cotas, key=lambda k: (-(ideal[k] - out[k]), k))[:resto]:
            out[k] += 1
        return out

    cotas = {}
    metade = {"close": total // 2, "distant": total - total // 2}
    for g in ("close", "distant"):
        disp = {(g, L): len(buckets[(g, L)]) for L in (1, 2, 3, 4)}
        cotas.update(encher(disp, metade[g]))

    q_blind = repartir(cotas, n_blind)
    q_dev = repartir({k: cotas[k] - q_blind[k] for k in cotas}, n_dev)

    blind, dev, train = [], [], []
    for k in sorted(cotas):
        xs = buckets[k][:cotas[k]]
        nb_, nd_ = q_blind[k], q_dev[k]
        blind += xs[:nb_]
        dev += xs[nb_:nb_ + nd_]
        train += xs[nb_ + nd_:]

    for nome_split, xs, n in (("train", train, n_train), ("dev", dev, n_dev), ("blind", blind, n_blind)):
        if len(xs) != n:
            raise SystemExit(f"ERRO: split {nome_split} tem {len(xs)} entidades; esperado {n}")
        if {x["n_tokens"] for x in xs} != {1, 2, 3, 4}:
            raise SystemExit(f"ERRO: split {nome_split} não cobre 1..4 tokens")
        if {x["grupo"] for x in xs} != {"close", "distant"}:
            raise SystemExit(f"ERRO: split {nome_split} não tem close e distant")
        rr.shuffle(xs)

    print("Cotas por (grupo, tokens):", dict(sorted(cotas.items())), flush=True)
    return train, dev, blind

TRAIN_ENT, DEV_ENT, BLIND_ENT = particionar_estratificado(pool)
tentativa_split = 1


def nomes(xs):
    return {x["nome"].casefold() for x in xs}

assert nomes(TRAIN_ENT).isdisjoint(nomes(DEV_ENT))
assert nomes(TRAIN_ENT).isdisjoint(nomes(BLIND_ENT))
assert nomes(DEV_ENT).isdisjoint(nomes(BLIND_ENT))

# Categoria/função/material são atribuídos só DEPOIS da partição, de modo determinístico.
def completar(entidades, offset):
    out = []
    for i, e in enumerate(entidades):
        z = dict(e)
        z["categoria"] = CATEGORIAS[(i + offset) % len(CATEGORIAS)]
        z["material_a"] = MATERIAIS[(2*i + offset) % len(MATERIAIS)]
        z["material_b"] = MATERIAIS[(2*i + offset + 1) % len(MATERIAIS)]
        out.append(z)
    return out

TRAIN_ENT = completar(TRAIN_ENT, 0)
DEV_ENT = completar(DEV_ENT, 1)
BLIND_ENT = completar(BLIND_ENT, 2)

# Congela o blind em cópia JSON antes de qualquer optimizer.
BLIND_CONGELADO = json.loads(json.dumps(BLIND_ENT, ensure_ascii=False))

def resumo_split(xs):
    return {
        "n": len(xs),
        "grupo": dict(Counter(x["grupo"] for x in xs)),
        "tokens": dict(sorted(Counter(x["n_tokens"] for x in xs).items())),
    }

print("split encontrado na tentativa", tentativa_split, flush=True)
print("train:", resumo_split(TRAIN_ENT), flush=True)
print("dev:  ", resumo_split(DEV_ENT), flush=True)
print("blind:", resumo_split(BLIND_CONGELADO), flush=True)
print("Interseções train/dev/blind = 0", flush=True)
print("PRECHECK 5/8 OK", flush=True)


In [ ]:
# Célula 6 — montar tarefas com formato REAL <usuario> ... <fim> <assistente>
print("PRECHECK 6/8 — tarefas", flush=True)

TAREFAS = ("copia", "categoria", "funcao", "invariancia", "composicao", "negacao")
POSICOES = ("inicio", "meio", "fim")
ENCHIMENTOS = [
    "Considere apenas os dados desta ficha.",
    "O catálogo experimental preserva a identidade dos objetos.",
    "A descrição seguinte é suficiente para responder.",
    "Não invente propriedades além das fornecidas.",
]

def E(texto: str):
    return enc(texto)

def contem_subspan(hay, needle):
    if not needle or len(needle) > len(hay):
        return False
    n = len(needle)
    return any(hay[i:i+n] == needle for i in range(len(hay)-n+1))

def add_part(ids_out, mark_out, ids_part, copiar=False):
    ids_out.extend(ids_part)
    mark_out.extend([bool(copiar)] * len(ids_part))

def resposta_para(ent, tarefa):
    nome = list(ent["ids"])
    cat = CAT_TXT[ent["categoria"]]
    fun = FUNCOES_TXT[ent["categoria"]]
    errada = CAT_TXT[next(c for c in CATEGORIAS if c != ent["categoria"])]

    out, mark = [], []
    if tarefa == "copia":
        add_part(out, mark, nome, True)
    elif tarefa == "categoria":
        add_part(out, mark, nome, True)
        add_part(out, mark, E(" é um "), False)
        add_part(out, mark, E(cat), False)
        add_part(out, mark, E("."), False)
    elif tarefa == "funcao":
        add_part(out, mark, nome, True)
        add_part(out, mark, E(" serve para "), False)
        add_part(out, mark, E(fun), False)
        add_part(out, mark, E("."), False)
    elif tarefa == "invariancia":
        add_part(out, mark, E("Não. "), False)
        add_part(out, mark, nome, True)
        add_part(out, mark, E(" continua sendo o mesmo objeto."), False)
    elif tarefa == "composicao":
        add_part(out, mark, nome, True)
        add_part(out, mark, E(" é um "), False)
        add_part(out, mark, E(cat), False)
        add_part(out, mark, E(" usado para "), False)
        add_part(out, mark, E(fun), False)
        add_part(out, mark, E("."), False)
    elif tarefa == "negacao":
        add_part(out, mark, E("Não. "), False)
        add_part(out, mark, nome, True)
        add_part(out, mark, E(" é um "), False)
        add_part(out, mark, E(cat), False)
        add_part(out, mark, E(", não um "), False)
        add_part(out, mark, E(errada), False)
        add_part(out, mark, E("."), False)
    else:
        raise ValueError(tarefa)

    out.append(ID_FIM)
    mark.append(False)
    return out, mark

def pergunta_para(ent, tarefa):
    nome = ent["nome"]
    errada = CAT_TXT[next(c for c in CATEGORIAS if c != ent["categoria"])]
    if tarefa == "copia":
        return "Repita exatamente o nome do objeto e nada além do nome."
    if tarefa == "categoria":
        return f"Qual é a categoria de {nome}? Preserve o nome exato."
    if tarefa == "funcao":
        return f"Qual é a função de {nome}? Preserve o nome exato."
    if tarefa == "invariancia":
        return f"Mudar o material de {ent['material_a']} para {ent['material_b']} muda a identidade? Use o nome exato."
    if tarefa == "composicao":
        return "Diga em uma frase o nome exato, a categoria e a função."
    if tarefa == "negacao":
        return f"{nome} é um {errada}? Corrija usando o nome exato."
    raise ValueError(tarefa)

def mensagem_ids(ent, tarefa, posicao):
    nome = list(ent["ids"])
    cat = CAT_TXT[ent["categoria"]]
    fun = FUNCOES_TXT[ent["categoria"]]

    # A definição é montada por partes para garantir que o span exato de nome esteja no prompt.
    definicao = []
    definicao += E("Existe um objeto inventado chamado ")
    definicao += nome
    definicao += E(". Ele é um ")
    definicao += E(cat)
    definicao += E(" usado para ")
    definicao += E(fun)
    definicao += E(".")

    q = E(" " + pergunta_para(ent, tarefa))
    if posicao == "inicio":
        msg = definicao + E(" " + ENCHIMENTOS[0]) + q
    elif posicao == "meio":
        msg = E(ENCHIMENTOS[1] + " ") + definicao + E(" " + ENCHIMENTOS[2]) + q
    elif posicao == "fim":
        msg = E(ENCHIMENTOS[2] + " " + ENCHIMENTOS[3] + " ") + definicao + q
    else:
        raise ValueError(posicao)

    if not contem_subspan(msg, nome):
        raise RuntimeError("span do nome desapareceu da mensagem")
    return msg

def montar_exemplo(ent, tarefa, posicao):
    msg = mensagem_ids(ent, tarefa, posicao)
    # Formato real de fonte_dialogo: <usuario> msg <fim> <assistente>
    prompt = [ID_USUARIO] + msg + [ID_FIM, ID_ASSISTENTE]
    resp, copy_resp = resposta_para(ent, tarefa)

    if prompt[-2:] != [ID_FIM, ID_ASSISTENTE]:
        raise RuntimeError("prompt não termina em <fim> <assistente>")
    if not contem_subspan(prompt, ent["ids"]):
        raise RuntimeError("nome não aparece como span exato no prompt")
    if len(prompt) + len(resp) > cfg.contexto:
        raise RuntimeError(
            f"exemplo excede contexto: {len(prompt)}+{len(resp)}>{cfg.contexto}"
        )

    return {
        "nome": ent["nome"],
        "nome_ids": list(ent["ids"]),
        "grupo": ent["grupo"],
        "n_tokens": int(ent["n_tokens"]),
        "categoria": ent["categoria"],
        "cat_ids": E(CAT_TXT[ent["categoria"]]),
        "fun_ids": E(FUNCOES_TXT[ent["categoria"]]),
        "tarefa": tarefa,
        "posicao": posicao,
        "prompt_ids": prompt,
        "resp_ids": resp,
        "copy_resp": copy_resp,
        "resposta": tokenizer.decode([x for x in resp if x != ID_FIM]),
        "no_copy": False,
    }

rr = random.Random(SEED + 2002)

TREINO = []
for i, ent in enumerate(TRAIN_ENT):
    for j, tarefa in enumerate(TAREFAS):
        pos = POSICOES[(i + j + rr.randrange(3)) % 3]
        TREINO.append(montar_exemplo(ent, tarefa, pos))
rr.shuffle(TREINO)

DEV_EX = []
for i, ent in enumerate(DEV_ENT):
    for j, tarefa in enumerate(("copia", "categoria", "funcao")):
        pos = POSICOES[(i + j) % 3]
        DEV_EX.append(montar_exemplo(ent, tarefa, pos))

BLIND_EX = []
for i, ent in enumerate(BLIND_CONGELADO):
    for j, tarefa in enumerate(TAREFAS):
        pos = POSICOES[(i + j) % 3]
        BLIND_EX.append(montar_exemplo(ent, tarefa, pos))

if len(BLIND_EX) != 600:
    raise SystemExit(f"ERRO: blind deveria ter 600 casos; tem {len(BLIND_EX)}")

# Exemplos no-copy: usados SOMENTE no treino/dev do gate; não entram no blind de 600.
NO_COPY_PARES = [
    ("Quanto é dois mais dois? Responda por extenso.", "quatro"),
    ("Qual cor resulta de azul com amarelo? Responda com uma palavra.", "verde"),
    ("Qual é o contrário de quente? Responda com uma palavra.", "frio"),
    ("Qual é o contrário de subir? Responda com uma palavra.", "descer"),
    ("Complete: manhã, tarde e ...", "noite"),
    ("Qual animal doméstico costuma miar?", "gato"),
    ("Qual astro ilumina o dia?", "sol"),
    ("Qual é o plural de casa?", "casas"),
    ("Qual estação vem depois do inverno?", "primavera"),
    ("Qual é o contrário de abrir?", "fechar"),
    ("Qual número vem depois de nove? Responda por extenso.", "dez"),
    ("Qual é a primeira vogal?", "a"),
]

def montar_no_copy(pergunta, resposta):
    # Aqui podemos usar diretamente a função real do repo.
    prompt = lp.fonte_dialogo(tokenizer, pergunta, (), cfg.contexto)
    resp = E(resposta) + [ID_FIM]
    if len(prompt) + len(resp) > cfg.contexto:
        raise RuntimeError("no-copy excedeu contexto")
    return {
        "nome": None,
        "nome_ids": [],
        "grupo": "no_copy",
        "n_tokens": 0,
        "categoria": None,
        "cat_ids": [],
        "fun_ids": [],
        "tarefa": "no_copy",
        "posicao": "na",
        "prompt_ids": list(prompt),
        "resp_ids": resp,
        "copy_resp": [False] * len(resp),
        "resposta": tokenizer.decode([x for x in resp if x != ID_FIM]),
        "no_copy": True,
    }

NO_COPY_EX = [montar_no_copy(q, a) for q, a in NO_COPY_PARES]

# Garantias anti-leak.
nomes_train = nomes(TRAIN_ENT)
nomes_dev = nomes(DEV_ENT)
nomes_blind = nomes(BLIND_CONGELADO)
assert nomes_train.isdisjoint(nomes_dev)
assert nomes_train.isdisjoint(nomes_blind)
assert nomes_dev.isdisjoint(nomes_blind)
assert all(ex["prompt_ids"][-2:] == [ID_FIM, ID_ASSISTENTE] for ex in TREINO[:100])
assert all(not ex["no_copy"] for ex in BLIND_EX)

print("treino:", len(TREINO), "dev:", len(DEV_EX), "blind:", len(BLIND_EX), "no-copy:", len(NO_COPY_EX), flush=True)
print("blind tarefas:", dict(Counter(x["tarefa"] for x in BLIND_EX)), flush=True)
print("blind posições:", dict(Counter(x["posicao"] for x in BLIND_EX)), flush=True)
print("blind grupos:", dict(Counter(x["grupo"] for x in BLIND_EX)), flush=True)
print("blind token lengths:", dict(sorted(Counter(x["n_tokens"] for x in BLIND_EX).items())), flush=True)
print("PRECHECK 6/8 OK", flush=True)


In [ ]:
# Célula 7 — batch, loss, geração e métricas auxiliares
print("PRECHECK 7/8 — loss/geração", flush=True)

def colar_batch(exemplos, device):
    fulls, labels_rows, copy_rows, prompt_lens = [], [], [], []
    for ex in exemplos:
        prompt = list(ex["prompt_ids"])
        resp = list(ex["resp_ids"])
        copy_resp = list(ex["copy_resp"])
        if len(resp) != len(copy_resp):
            raise ValueError("resp_ids/copy_resp com tamanhos diferentes")
        full = prompt + resp
        if len(full) > cfg.contexto:
            raise ValueError("batch contém sequência acima do contexto")

        labels = [-100] * len(prompt) + resp
        copy_target = [False] * len(prompt) + copy_resp

        fulls.append(full)
        labels_rows.append(labels)
        copy_rows.append(copy_target)
        prompt_lens.append(len(prompt))

    tmax = max(map(len, fulls))
    b = len(fulls)
    ids = torch.full((b, tmax), ID_PAD, dtype=torch.long, device=device)
    labels = torch.full((b, tmax), -100, dtype=torch.long, device=device)
    copy_targets = torch.zeros((b, tmax), dtype=torch.bool, device=device)

    for i, (full, lab, cpy) in enumerate(zip(fulls, labels_rows, copy_rows)):
        n = len(full)
        ids[i, :n] = torch.tensor(full, dtype=torch.long, device=device)
        labels[i, :n] = torch.tensor(lab, dtype=torch.long, device=device)
        copy_targets[i, :n] = torch.tensor(cpy, dtype=torch.bool, device=device)

    prompt_lens_t = torch.tensor(prompt_lens, dtype=torch.long, device=device)
    return ids, labels, copy_targets, prompt_lens_t

def perda_batch(pointer_obj, exemplos, device):
    ids, labels, copy_targets, prompt_lens = colar_batch(exemplos, device)
    p_final, _, p_copy, p_gen, _, _, src = distribuicao_final(pointer_obj, ids, prompt_lens)

    # hidden position t predicts token t+1
    pred = p_final[:, :-1, :]
    copy_pred = p_copy[:, :-1, :]
    gate_pred = p_gen[:, :-1, 0]
    gold = labels[:, 1:]
    supervised = gold.ne(-100)
    copy_gold_mask = copy_targets[:, 1:] & supervised

    if not supervised.any():
        raise RuntimeError("batch sem tokens supervisionados")

    safe_gold = gold.clamp_min(0)
    prob_gold = pred.gather(-1, safe_gold.unsqueeze(-1)).squeeze(-1)
    mix_nll = -torch.log(prob_gold.clamp_min(1e-8))
    loss_mix = mix_nll[supervised].mean()

    # Copy CE SOMENTE nos tokens explicitamente marcados como span da entidade.
    if copy_gold_mask.any():
        copy_prob_gold = copy_pred.gather(-1, safe_gold.unsqueeze(-1)).squeeze(-1)
        copy_nll = -torch.log(copy_prob_gold.clamp_min(1e-8))
        loss_copy = copy_nll[copy_gold_mask].mean()
        # Se isso for zero, há bug de source mask/span.
        if not torch.isfinite(loss_copy):
            raise RuntimeError("loss_copy não finita")
    else:
        loss_copy = torch.zeros((), device=device)

    # Gate supervisionado: copy span -> p_gen=0; demais tokens da resposta -> p_gen=1.
    gate_target = (~copy_gold_mask).float()
    gate_bce_all = F.binary_cross_entropy(
        gate_pred.clamp(1e-6, 1 - 1e-6), gate_target, reduction="none"
    )
    loss_gate = gate_bce_all[supervised].mean()

    loss = loss_mix + 0.50 * loss_copy + 0.10 * loss_gate

    with torch.no_grad():
        pg_copy = gate_pred[copy_gold_mask].mean().item() if copy_gold_mask.any() else None
        gen_mask = supervised & ~copy_gold_mask
        pg_gen = gate_pred[gen_mask].mean().item() if gen_mask.any() else None
        src_frac = src.float().mean().item()

    metricas = {
        "loss_mix": float(loss_mix.detach().item()),
        "loss_copy": float(loss_copy.detach().item()),
        "loss_gate": float(loss_gate.detach().item()),
        "p_gen_copy": None if pg_copy is None else float(pg_copy),
        "p_gen_generate": None if pg_gen is None else float(pg_gen),
        "source_mask_fraction": float(src_frac),
    }
    return loss, metricas

@torch.no_grad()
def probs_base(ids):
    h = estados_congelados(ids)
    logits = F.linear(h, modelo.embedding.weight)
    p = torch.softmax(logits, dim=-1)
    if PROIBIDOS_GERACAO:
        p[:, :, PROIBIDOS_GERACAO] = 0.0
        p = p / p.sum(dim=-1, keepdim=True).clamp_min(1e-8)
    return p

@torch.no_grad()
def gerar_base(prompt_ids, max_novos=28):
    ids = list(prompt_ids)
    gerados = []
    limite = min(max_novos, max(0, cfg.contexto - len(ids)))
    for _ in range(limite):
        x = torch.tensor([ids], dtype=torch.long, device=dispositivo)
        p = probs_base(x)[0, -1]
        nxt = int(torch.argmax(p).item())
        if nxt == ID_FIM:
            return gerados, True
        if nxt in PROIBIDOS_GERACAO:
            raise RuntimeError("base tentou emitir token especial proibido após mascaramento")
        gerados.append(nxt)
        ids.append(nxt)
    return gerados, False

@torch.no_grad()
def gerar_pointer(pointer_obj, prompt_ids, max_novos=28):
    ids = list(prompt_ids)
    prompt_len = len(prompt_ids)
    gerados = []
    pgens = []
    limite = min(max_novos, max(0, cfg.contexto - prompt_len))
    for _ in range(limite):
        x = torch.tensor([ids], dtype=torch.long, device=dispositivo)
        pl = torch.tensor([prompt_len], dtype=torch.long, device=dispositivo)
        p_final, _, _, p_gen, _, _, src = distribuicao_final(pointer_obj, x, pl)

        # Teste anti-vazamento: qualquer token gerado fica FORA da source mask.
        if len(ids) > prompt_len and bool(src[0, prompt_len:].any().item()):
            raise RuntimeError("BUG: token gerado entrou na source mask do pointer")

        pgens.append(float(p_gen[0, -1, 0].item()))
        nxt = int(torch.argmax(p_final[0, -1]).item())
        if nxt == ID_FIM:
            return gerados, True, (sum(pgens) / len(pgens) if pgens else None)
        if nxt in PROIBIDOS_GERACAO:
            raise RuntimeError("pointer tentou emitir especial proibido")
        gerados.append(nxt)
        ids.append(nxt)
    return gerados, False, (sum(pgens) / len(pgens) if pgens else None)

def token_edit_distance(a, b):
    a, b = list(a), list(b)
    if not a:
        return len(b)
    if not b:
        return len(a)
    prev = list(range(len(b)+1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(
                cur[-1] + 1,
                prev[j] + 1,
                prev[j-1] + (x != y),
            ))
        prev = cur
    return prev[-1]

def min_span_edit_distance(saida, nome_ids):
    """Distância de edição do nome para o MELHOR span da saída, não para a resposta inteira."""
    if not nome_ids:
        return None
    if not saida:
        return len(nome_ids)
    n = len(nome_ids)
    melhor = n + len(saida)
    lo = max(1, n - 1)
    hi = min(len(saida), n + 1)
    for tam in range(lo, hi + 1):
        for i in range(0, len(saida) - tam + 1):
            melhor = min(melhor, token_edit_distance(saida[i:i+tam], nome_ids))
    return melhor

def max_novos_ex(ex):
    return {
        "copia": 8,
        "categoria": 18,
        "funcao": 22,
        "invariancia": 24,
        "composicao": 28,
        "negacao": 28,
        "no_copy": 12,
    }.get(ex["tarefa"], 24)

def sinais_semanticos(ex, gerados):
    if ex["no_copy"]:
        return None
    nome_ok = contem_subspan(gerados, ex["nome_ids"])
    cat_ok = contem_subspan(gerados, ex["cat_ids"]) if ex["cat_ids"] else False
    fun_ok = contem_subspan(gerados, ex["fun_ids"]) if ex["fun_ids"] else False
    texto = tokenizer.decode(gerados).casefold()
    neg = ("não" in texto) or ("nao" in texto)

    if ex["tarefa"] == "copia":
        return gerados == ex["nome_ids"]
    if ex["tarefa"] == "categoria":
        return nome_ok and cat_ok
    if ex["tarefa"] == "funcao":
        return nome_ok and fun_ok
    if ex["tarefa"] == "invariancia":
        return nome_ok and neg
    if ex["tarefa"] == "composicao":
        return nome_ok and cat_ok and fun_ok
    if ex["tarefa"] == "negacao":
        return nome_ok and cat_ok and neg
    return False

print("PRECHECK 7/8 OK", flush=True)


In [ ]:
# Célula 8 — SANITY + smoke test adversarial antes de qualquer treino real
print("PRECHECK 8/8 — SANITY/SMOKE", flush=True)

# Estrutura dos dados
assert len(TRAIN_ENT) == 480 and len(DEV_ENT) == 80 and len(BLIND_CONGELADO) == 100
assert len(BLIND_EX) == 600
assert nomes(TRAIN_ENT).isdisjoint(nomes(DEV_ENT))
assert nomes(TRAIN_ENT).isdisjoint(nomes(BLIND_CONGELADO))
assert nomes(DEV_ENT).isdisjoint(nomes(BLIND_CONGELADO))
assert all(ex["prompt_ids"][-2:] == [ID_FIM, ID_ASSISTENTE] for ex in BLIND_EX)
assert all(len(ex["prompt_ids"]) + len(ex["resp_ids"]) <= cfg.contexto for ex in BLIND_EX)

pointer_sanity = PointerCopy(cfg.dimensao, cfg.vocabulario, rank=RANK).to(dispositivo)
opt_sanity = torch.optim.AdamW(pointer_sanity.parameters(), lr=1e-3, weight_decay=0.0)

rr_s = random.Random(SEED + 3003)
copy_pool = [x for x in TREINO if x["tarefa"] in ("copia", "categoria", "funcao")]
no_copy_pool = NO_COPY_EX

for passo in range(1, 6):
    batch = rr_s.sample(copy_pool, 8) + [rr_s.choice(no_copy_pool) for _ in range(4)]
    rr_s.shuffle(batch)
    opt_sanity.zero_grad(set_to_none=True)
    loss_s, met_s = perda_batch(pointer_sanity, batch, dispositivo)
    if not torch.isfinite(loss_s):
        raise SystemExit(f"ERRO SANITY: loss não finita no passo {passo}")
    loss_s.backward()

    soma_grad = 0.0
    for nome, p in pointer_sanity.named_parameters():
        if p.grad is None:
            raise SystemExit(f"ERRO SANITY: pointer sem gradiente em {nome}")
        if not torch.isfinite(p.grad).all():
            raise SystemExit(f"ERRO SANITY: gradiente NaN/Inf em {nome}")
        soma_grad += float(p.grad.abs().sum().item())

    if soma_grad <= 0:
        raise SystemExit("ERRO SANITY: todos os gradientes do pointer são zero")

    for nome, p in modelo.named_parameters():
        if p.requires_grad:
            raise SystemExit(f"ERRO SANITY: base descongelada em {nome}")
        if p.grad is not None:
            raise SystemExit(f"ERRO SANITY: base recebeu gradiente em {nome}")

    torch.nn.utils.clip_grad_norm_(pointer_sanity.parameters(), 1.0)
    opt_sanity.step()
    print(
        f"  sanity step {passo}/5 loss={loss_s.item():.4f} "
        f"mix={met_s['loss_mix']:.4f} copy={met_s['loss_copy']:.4f} "
        f"gate={met_s['loss_gate']:.4f}",
        flush=True,
    )

# Geração autoregressiva real para pegar shape/source-mask bugs que teacher forcing não pega.
teste_ex = next(x for x in DEV_EX if x["tarefa"] == "copia")
gerados_s, _, pg_s = gerar_pointer(pointer_sanity, teste_ex["prompt_ids"], max_novos=8)
print("  smoke geração:", tokenizer.decode(gerados_s), "p_gen_médio=", pg_s, flush=True)

HASH_SANITY = fingerprint_modulo(modelo)
if HASH_SANITY != HASH_ANTES:
    raise SystemExit(
        "ERRO SANITY: hash da base mudou.\n"
        f"antes={HASH_ANTES}\ndepois={HASH_SANITY}"
    )

del pointer_sanity, opt_sanity
torch.cuda.empty_cache()

# Recria pointer LIMPO: os 5 steps acima não contaminam o experimento.
pointer = PointerCopy(cfg.dimensao, cfg.vocabulario, rank=RANK).to(dispositivo)
print("SANITY CHECK PASSOU", flush=True)
print("Base intacta; gradiente só no pointer; source-mask bloqueia resposta; geração não quebrou.", flush=True)
print("\n✅ PRECHECK COMPLETO — pronto para medir baseline e iniciar experimento.", flush=True)


In [ ]:
# Célula 9 — avaliação congelada: funções de avaliação + BASELINE antes do treino

def avaliar_blind(modo, exemplos, pointer_obj=None, rotulo="eval"):
    if modo not in ("base", "pointer"):
        raise ValueError(modo)
    print(f"Avaliando {rotulo}: {len(exemplos)} casos", flush=True)
    linhas = []
    t0 = time.time()

    for i, ex in enumerate(exemplos, 1):
        if modo == "base":
            gerados, terminou = gerar_base(ex["prompt_ids"], max_novos=max_novos_ex(ex))
            pgen_medio = None
        else:
            gerados, terminou, pgen_medio = gerar_pointer(
                pointer_obj, ex["prompt_ids"], max_novos=max_novos_ex(ex)
            )

        nome_ok = contem_subspan(gerados, ex["nome_ids"])
        pure = ex["tarefa"] == "copia" and gerados == ex["nome_ids"]
        sem_ok = bool(sinais_semanticos(ex, gerados))
        ed = min_span_edit_distance(gerados, ex["nome_ids"])

        linhas.append({
            "tarefa": ex["tarefa"],
            "nome": ex["nome"],
            "grupo": ex["grupo"],
            "posicao": ex["posicao"],
            "n_tokens": ex["n_tokens"],
            "entity_exact": bool(nome_ok),
            "pure_copy_exact": bool(pure),
            "semantic_ok": bool(sem_ok),
            "edit_distance_tokens": int(ed),
            "edit_distance_le1": bool(ed <= 1),
            "terminou": bool(terminou),
            "p_gen_medio": pgen_medio,
            "saida": tokenizer.decode(gerados),
        })

        if i == 1 or i % 50 == 0 or i == len(exemplos):
            print(f"  {rotulo}: {i}/{len(exemplos)} ({time.time()-t0:.1f}s)", flush=True)

    return linhas

def resumir(linhas):
    def subset(pred):
        return [r for r in linhas if pred(r)]
    def media_bool(xs, chave):
        return None if not xs else sum(bool(x[chave]) for x in xs) / len(xs)
    def entity_rate(pred):
        xs = subset(pred)
        return media_bool(xs, "entity_exact")
    def sem_rate(tarefa):
        xs = subset(lambda r: r["tarefa"] == tarefa)
        return media_bool(xs, "semantic_ok")

    copia = subset(lambda r: r["tarefa"] == "copia")
    eds = [r["edit_distance_tokens"] for r in linhas]
    out = {
        "n": len(linhas),
        "exact_entity_match": media_bool(linhas, "entity_exact"),
        "pure_copy_exact_match": media_bool(copia, "pure_copy_exact"),
        "edit_distance_mean_tokens": sum(eds)/len(eds) if eds else None,
        "edit_distance_le1": sum(r["edit_distance_le1"] for r in linhas)/len(linhas) if linhas else None,
        "lexical_close": entity_rate(lambda r: r["grupo"] == "close"),
        "lexical_distant": entity_rate(lambda r: r["grupo"] == "distant"),
        "posicao_inicio": entity_rate(lambda r: r["posicao"] == "inicio"),
        "posicao_meio": entity_rate(lambda r: r["posicao"] == "meio"),
        "posicao_fim": entity_rate(lambda r: r["posicao"] == "fim"),
        "token_len_1": entity_rate(lambda r: r["n_tokens"] == 1),
        "token_len_2": entity_rate(lambda r: r["n_tokens"] == 2),
        "token_len_3": entity_rate(lambda r: r["n_tokens"] == 3),
        "token_len_4": entity_rate(lambda r: r["n_tokens"] == 4),
        "categoria": sem_rate("categoria"),
        "funcao": sem_rate("funcao"),
        "invariancia": sem_rate("invariancia"),
        "composicao": sem_rate("composicao"),
        "negacao": sem_rate("negacao"),
        "termination_rate": media_bool(linhas, "terminou"),
        "p_gen_medio": (
            sum(r["p_gen_medio"] for r in linhas if r["p_gen_medio"] is not None)
            / max(1, sum(r["p_gen_medio"] is not None for r in linhas))
            if any(r["p_gen_medio"] is not None for r in linhas) else None
        ),
    }
    sem_vals = [out[k] for k in ("categoria","funcao","invariancia","composicao","negacao") if out[k] is not None]
    out["semantica_geral"] = sum(sem_vals)/len(sem_vals) if sem_vals else None
    return out

@torch.no_grad()
def preservacao_next_token(pointer_obj):
    """
    Compara BASE vs POINTER no primeiro token de respostas genéricas sem entidade sintética.
    É uma métrica extra; não substitui os gates semânticos do blind.
    """
    prompts = [
        "Explique em uma frase o que é gravidade.",
        "Qual é a função de uma cadeira?",
        "O que significa memória?",
        "Diga uma característica da água.",
        "O que é uma estrela?",
        "Qual é a função de um recipiente?",
        "Explique o que é temperatura.",
        "O que significa aprender?",
        "Diga uma função de uma ferramenta.",
        "O que é uma hipótese?",
        "Explique o que é energia.",
        "Qual é a diferença entre fato e opinião?",
    ]
    acordos, tvs = [], []
    for msg in prompts:
        prompt = lp.fonte_dialogo(tokenizer, msg, (), cfg.contexto)
        x = torch.tensor([prompt], dtype=torch.long, device=dispositivo)
        pb = probs_base(x)[0, -1]
        pl = torch.tensor([len(prompt)], dtype=torch.long, device=dispositivo)
        pp, *_ = distribuicao_final(pointer_obj, x, pl)
        pp = pp[0, -1]
        acordos.append(int(torch.argmax(pb)) == int(torch.argmax(pp)))
        tvs.append(0.5 * torch.abs(pb - pp).sum().item())
    return {
        "top1_agreement": sum(acordos)/len(acordos),
        "mean_total_variation": sum(tvs)/len(tvs),
        "n": len(prompts),
    }

print("\n=== BASELINE CONGELADO — antes de qualquer treino do pointer ===", flush=True)
BASELINE_LINHAS = avaliar_blind("base", BLIND_EX, rotulo="baseline")
BASELINE = resumir(BASELINE_LINHAS)
print(json.dumps(BASELINE, ensure_ascii=False, indent=2), flush=True)


In [ ]:
# Célula 10 — treino do pointer/gate: até 1000 passos, com no-copy e early stop composto
PASSOS = 1000
BATCH = 16
N_NO_COPY = 4          # 25% do batch protege p_gen / geração livre
LR = 2e-3
EVAL_CADA = 100
PACIENCIA = 4

rr_train = random.Random(SEED + 4004)
otimizador = torch.optim.AdamW(pointer.parameters(), lr=LR, weight_decay=0.01)

dev_copy = [x for x in DEV_EX if x["tarefa"] == "copia"][:16]
dev_func = [x for x in DEV_EX if x["tarefa"] == "funcao"][:16]

@torch.no_grad()
def dev_monitor(pointer_obj):
    pointer_obj.eval()

    ac_copy = 0
    for ex in dev_copy:
        g, _, _ = gerar_pointer(pointer_obj, ex["prompt_ids"], max_novos=8)
        ac_copy += (g == ex["nome_ids"])
    copy_acc = ac_copy / max(1, len(dev_copy))

    ac_func = 0
    for ex in dev_func:
        g, _, _ = gerar_pointer(pointer_obj, ex["prompt_ids"], max_novos=22)
        ac_func += bool(sinais_semanticos(ex, g))
    func_acc = ac_func / max(1, len(dev_func))

    pres = preservacao_next_token(pointer_obj)
    # Score não olha só copy: também recompensa função e preservação da base.
    score = copy_acc + 0.25 * func_acc + 0.25 * pres["top1_agreement"]
    pointer_obj.train()
    return {
        "copy_exact": copy_acc,
        "funcao": func_acc,
        "preservation_top1": pres["top1_agreement"],
        "preservation_tv": pres["mean_total_variation"],
        "score": score,
    }

historico = []
melhor_score = -1.0
melhor_state = None
sem_melhora = 0
passos_feitos = 0
t0 = time.time()

entity_batch_n = BATCH - N_NO_COPY
print(
    f"Iniciando treino real: até {PASSOS} passos | batch={BATCH} "
    f"({entity_batch_n} entidade + {N_NO_COPY} no-copy) | lr={LR}",
    flush=True,
)

pointer.train()
for passo in range(1, PASSOS + 1):
    batch = rr_train.sample(TREINO, entity_batch_n)
    batch += [rr_train.choice(NO_COPY_EX) for _ in range(N_NO_COPY)]
    rr_train.shuffle(batch)

    otimizador.zero_grad(set_to_none=True)
    loss, met = perda_batch(pointer, batch, dispositivo)
    if not torch.isfinite(loss):
        raise SystemExit(f"ERRO: loss NaN/Inf no passo {passo}")

    loss.backward()
    for nome, p in pointer.named_parameters():
        if p.grad is None or not torch.isfinite(p.grad).all():
            raise SystemExit(f"ERRO: gradiente inválido no pointer: {nome}")

    torch.nn.utils.clip_grad_norm_(pointer.parameters(), 1.0)
    otimizador.step()
    passos_feitos = passo

    if passo == 1 or passo % 25 == 0:
        print(
            f"PASSO {passo:4d}/{PASSOS} "
            f"loss={loss.item():.4f} mix={met['loss_mix']:.4f} "
            f"copy={met['loss_copy']:.4f} gate={met['loss_gate']:.4f} "
            f"p_gen(copy)={met['p_gen_copy']} p_gen(gen)={met['p_gen_generate']}",
            flush=True,
        )

    if passo == 1 or passo % EVAL_CADA == 0:
        dm = dev_monitor(pointer)
        registro = {
            "passo": passo,
            "loss": float(loss.item()),
            **met,
            **{f"dev_{k}": float(v) for k, v in dm.items()},
            "segundos": round(time.time() - t0, 1),
        }
        historico.append(registro)
        print(
            f"  DEV passo={passo} copy={dm['copy_exact']:.3f} "
            f"funcao={dm['funcao']:.3f} preserve_top1={dm['preservation_top1']:.3f} "
            f"TV={dm['preservation_tv']:.3f} score={dm['score']:.3f}",
            flush=True,
        )

        if dm["score"] > melhor_score + 1e-4:
            melhor_score = dm["score"]
            melhor_state = {k: v.detach().cpu().clone() for k, v in pointer.state_dict().items()}
            sem_melhora = 0
        else:
            sem_melhora += 1
            print(f"  sem melhora composta: {sem_melhora}/{PACIENCIA}", flush=True)

        if passo >= 300 and sem_melhora >= PACIENCIA:
            print("EARLY STOP: score composto do dev parou de melhorar.", flush=True)
            break

    if passo % 200 == 0:
        h = fingerprint_modulo(modelo)
        if h != HASH_ANTES:
            raise SystemExit("ERRO: hash da base mudou durante o treino. Experimento inválido.")

if melhor_state is None:
    raise SystemExit("ERRO: nenhum estado válido do pointer foi registrado.")

pointer.load_state_dict(melhor_state)
pointer.eval()

HASH_APOS_TREINO = fingerprint_modulo(modelo)
if HASH_APOS_TREINO != HASH_ANTES:
    raise SystemExit(
        "ERRO: Transformer mudou durante o treino.\n"
        f"antes={HASH_ANTES}\ndepois={HASH_APOS_TREINO}"
    )

print(
    f"Treino finalizado: {passos_feitos} passos executados; "
    f"melhor_score_dev={melhor_score:.4f}; base intacta.",
    flush=True,
)


In [ ]:
# Célula 11 — avaliação FINAL no MESMO blind congelado + preservação extra
print("\n=== POINTER FINAL — mesmo blind congelado ===", flush=True)

FINAL_LINHAS = avaliar_blind("pointer", BLIND_EX, pointer_obj=pointer, rotulo="pointer_final")
FINAL = resumir(FINAL_LINHAS)
PRESERVACAO = preservacao_next_token(pointer)

print("BASELINE:", json.dumps(BASELINE, ensure_ascii=False, indent=2), flush=True)
print("FINAL:", json.dumps(FINAL, ensure_ascii=False, indent=2), flush=True)
print("PRESERVAÇÃO EXTRA:", json.dumps(PRESERVACAO, ensure_ascii=False, indent=2), flush=True)

HASH_APOS_AVALIACAO = fingerprint_modulo(modelo)
if HASH_APOS_AVALIACAO != HASH_ANTES:
    raise SystemExit("ERRO: hash da base mudou durante avaliação.")


In [ ]:
# Célula 12 — gates congelados + salvamento no Drive
print("PÓS-CHECK — gates e salvamento", flush=True)

HASH_DEPOIS = fingerprint_modulo(modelo)
BASE_UNCHANGED = HASH_DEPOIS == HASH_ANTES
if not BASE_UNCHANGED:
    raise SystemExit("ERRO: base mudou. Nada será salvo como resultado válido.")

def f(d, k):
    v = d.get(k)
    return None if v is None else float(v)

close_f = f(FINAL, "lexical_close")
dist_f = f(FINAL, "lexical_distant")
pos_vals = [v for v in (f(FINAL,"posicao_inicio"), f(FINAL,"posicao_meio"), f(FINAL,"posicao_fim")) if v is not None]
fun_b, fun_f = f(BASELINE, "funcao"), f(FINAL, "funcao")
sem_b, sem_f = f(BASELINE, "semantica_geral"), f(FINAL, "semantica_geral")

GATES = {
    "copy_exact>=0.70": f(FINAL, "pure_copy_exact_match") is not None and f(FINAL, "pure_copy_exact_match") >= 0.70,
    "lexically_distant>=0.65": dist_f is not None and dist_f >= 0.65,
    "abs(close-distant)<=0.10": close_f is not None and dist_f is not None and abs(close_f-dist_f) <= 0.10,
    "max(position)-min(position)<=0.10": len(pos_vals) == 3 and (max(pos_vals)-min(pos_vals)) <= 0.10,
    "regressao_funcao<=5pp": fun_b is not None and fun_f is not None and (fun_b-fun_f) <= 0.05 + 1e-9,
    "regressao_semantica_geral<=2pp": sem_b is not None and sem_f is not None and (sem_b-sem_f) <= 0.02 + 1e-9,
    "transformer_hash_igual": BASE_UNCHANGED,
}
GATE_PASS = all(GATES.values())

EXTRA_CHECKS = {
    "preservation_top1>=0.90": PRESERVACAO["top1_agreement"] >= 0.90,
    "preservation_mean_TV<=0.15": PRESERVACAO["mean_total_variation"] <= 0.15,
}
# Extra checks são informativos; não alteram silenciosamente os gates definidos no protocolo.

def delta(k):
    a, b = BASELINE.get(k), FINAL.get(k)
    return None if a is None or b is None else float(b-a)

resultado = {
    "protocol": "pointer_copy_causal_v2",
    "created_at": datetime.now().isoformat(),
    "repo_commit": commit,
    "base_checkpoint": str(BASE_RUN),
    "base_phase": "05_05_dialogo_estado",
    "base_file_sha256": SHA_PESOS,
    "tokenizer_sha256": SHA_TOKENIZER,
    "base_hash_before": HASH_ANTES,
    "base_hash_after": HASH_DEPOIS,
    "base_unchanged": BASE_UNCHANGED,
    "base_parameters": int(N_BASE),
    "pointer_parameters": int(N_POINTER),
    "pointer_rank": int(RANK),
    "seed": int(SEED),
    "train_steps_requested": int(PASSOS),
    "train_steps_done": int(passos_feitos),
    "train_history": historico,
    "splits": {
        "train_entities": resumo_split(TRAIN_ENT),
        "dev_entities": resumo_split(DEV_ENT),
        "blind_entities": resumo_split(BLIND_CONGELADO),
        "blind_cases": len(BLIND_EX),
        "overlap_train_dev": len(nomes(TRAIN_ENT) & nomes(DEV_ENT)),
        "overlap_train_blind": len(nomes(TRAIN_ENT) & nomes(BLIND_CONGELADO)),
        "overlap_dev_blind": len(nomes(DEV_ENT) & nomes(BLIND_CONGELADO)),
    },
    "baseline": BASELINE,
    "final": FINAL,
    "preservation_extra": PRESERVACAO,
    "delta_baseline_to_final": {
        k: delta(k) for k in (
            "exact_entity_match", "pure_copy_exact_match", "lexical_close",
            "lexical_distant", "categoria", "funcao", "invariancia",
            "composicao", "negacao", "semantica_geral",
            "edit_distance_mean_tokens",
        )
    },
    "by_skill": {
        "copia": FINAL["pure_copy_exact_match"],
        "categoria": FINAL["categoria"],
        "funcao": FINAL["funcao"],
        "invariancia": FINAL["invariancia"],
        "composicao": FINAL["composicao"],
        "negacao": FINAL["negacao"],
    },
    "by_position": {
        "inicio": FINAL["posicao_inicio"],
        "meio": FINAL["posicao_meio"],
        "fim": FINAL["posicao_fim"],
    },
    "by_token_length": {
        "1": FINAL["token_len_1"],
        "2": FINAL["token_len_2"],
        "3": FINAL["token_len_3"],
        "4": FINAL["token_len_4"],
    },
    "gates": GATES,
    "extra_checks": EXTRA_CHECKS,
    "gate_pass": bool(GATE_PASS),
    "ativo_no_chat": False,
    "promovido": False,
    "samples_baseline": BASELINE_LINHAS[:24],
    "samples_final": FINAL_LINHAS[:24],
}

config_exp = {
    "protocol": "pointer_copy_causal_v2",
    "seed": SEED,
    "rank": RANK,
    "p_gen_init": 0.90,
    "batch": BATCH,
    "no_copy_per_batch": N_NO_COPY,
    "lr": LR,
    "steps_requested": PASSOS,
    "steps_done": passos_feitos,
    "loss": "mixture_ce + 0.50*entity_copy_ce + 0.10*gate_bce",
    "source_copy_policy": "prompt_original_only_excluding_specials",
    "prompt_format": "<usuario> mensagem <fim> <assistente>",
    "blind_frozen_before_training": True,
    "ativo_no_chat": False,
    "promovido": False,
}

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
DEST_ROOT = DRIVE_CHECKPOINTS / "experimento_pointer_copy"
DEST_ROOT.mkdir(parents=True, exist_ok=True)
DEST = DEST_ROOT / f"pointer_copy_{stamp}"
DEST.mkdir(parents=False, exist_ok=False)

torch.save(
    {
        "state_dict": {k: v.detach().cpu() for k, v in pointer.state_dict().items()},
        "rank": RANK,
        "dimensao": cfg.dimensao,
        "vocabulario": cfg.vocabulario,
        "base_checkpoint": str(BASE_RUN),
        "base_hash": HASH_DEPOIS,
        "base_file_sha256": SHA_PESOS,
        "tokenizer_sha256": SHA_TOKENIZER,
        "ativo_no_chat": False,
        "promovido": False,
    },
    DEST / "pointer_head.pt",
)

(DEST / "resultado_pointer_copy.json").write_text(
    json.dumps(resultado, ensure_ascii=False, indent=2), encoding="utf-8"
)
(DEST / "config_experimento.json").write_text(
    json.dumps(config_exp, ensure_ascii=False, indent=2), encoding="utf-8"
)
(DEST / "tokenizer.sha256.txt").write_text(SHA_TOKENIZER + "\n", encoding="utf-8")

readme = f"""CRIVO — pointer_copy_causal_v2
base: {BASE_RUN}
repo_commit: {commit}
base_file_sha256: {SHA_PESOS}
base_hash_before: {HASH_ANTES}
base_hash_after: {HASH_DEPOIS}
base_unchanged: {BASE_UNCHANGED}
pointer_params: {N_POINTER}
steps_done: {passos_feitos}

baseline pure_copy_exact: {BASELINE['pure_copy_exact_match']}
final pure_copy_exact: {FINAL['pure_copy_exact_match']}
baseline funcao: {BASELINE['funcao']}
final funcao: {FINAL['funcao']}
baseline semantica_geral: {BASELINE['semantica_geral']}
final semantica_geral: {FINAL['semantica_geral']}

gates:
{json.dumps(GATES, ensure_ascii=False, indent=2)}

extra_checks:
{json.dumps(EXTRA_CHECKS, ensure_ascii=False, indent=2)}

gate_pass: {GATE_PASS}
ativo_no_chat: false
promovido: false

OBS: pesos do Transformer não foram duplicados. pointer_head.pt depende do checkpoint-base acima.
"""
(DEST / "README_resultado.txt").write_text(readme, encoding="utf-8")

# Verifica que os arquivos realmente foram persistidos.
esperados_saida = [
    DEST/"pointer_head.pt",
    DEST/"resultado_pointer_copy.json",
    DEST/"config_experimento.json",
    DEST/"tokenizer.sha256.txt",
    DEST/"README_resultado.txt",
]
for p in esperados_saida:
    if not p.is_file() or p.stat().st_size == 0:
        raise SystemExit(f"ERRO de persistência: {p} não foi salvo corretamente.")

print("\n================ RESULTADO ================", flush=True)
print(readme, flush=True)
print("SALVO EM:", DEST, flush=True)
print("EXPERIMENTO POINTER COPY FINALIZADO", flush=True)
print(f"gate_pass={GATE_PASS} | ativo_no_chat=false | promovido=false", flush=True)
